In [1]:
# NB12_collinearity_diagnostics.ipynb

# ---------------------------------------------------------------------------
# NB12 - Multicollinearity Diagnostics for the System Credit Grading Model
#
# Purpose:
#   Reviewer comment (Section 3.2): the system-level logistic regression is
#   developed on 64 financial variables, but no multicollinearity diagnostics
#   are reported. Highly correlated predictors inflate standard errors and
#   destabilise coefficient estimates. This notebook quantifies collinearity
#   among the 64 predictors via (i) Variance Inflation Factors (VIF) and
#   (ii) an absolute-correlation clustermap, and summarises how many features
#   exceed the conventional VIF thresholds of 5 and 10.
#
# Design note:
#   The system grading model (NB03) scores default probability from the full
#   financial feature set. Discrimination (ROC-AUC), not inferential
#   interpretation of individual coefficients, is its purpose; VIF is reported
#   here to document the structure of the predictor space, and the paper notes
#   that L2-regularised logistic regression (C = 0.1) and the tree-based
#   models are robust to the collinearity that VIF reveals.
#
# Input:   ../data/processed/polish_master.parquet
# Output:  ../results/tables/19_vif.csv
#          ../results/tables/19b_collinearity_summary.csv
#          ../results/figures/29_vif_top.{png,pdf}
#          ../results/figures/30_correlation_clustermap.{png,pdf}
# ---------------------------------------------------------------------------

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

warnings.filterwarnings("ignore")

# ---- Grayscale house style (seaborn), high-resolution, caption-free -------
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({
    "figure.dpi"        : 120,
    "savefig.dpi"       : 600,
    "font.family"       : "serif",
    "axes.edgecolor"    : "black",
    "axes.linewidth"    : 0.8,
    "axes.grid"         : True,
    "grid.color"        : "0.85",
    "grid.linewidth"    : 0.5,
})
GREY_DARK, GREY_MID, GREY_LIGHT = "0.20", "0.45", "0.70"

FIG_DIR   = "../results/figures/"
TABLE_DIR = "../results/tables/"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)

def savefig(fig, name):
    # Save every figure as both 600-dpi PNG and vector PDF, no caption/title.
    fig.savefig(FIG_DIR + name + ".png", dpi=600, bbox_inches="tight")
    fig.savefig(FIG_DIR + name + ".pdf", bbox_inches="tight")
    plt.close(fig)

# ---- 1. Load data and reconstruct the system model feature matrix ---------
master = pd.read_parquet("../data/processed/polish_master.parquet")
EXCLUDE_COLS = {"default", "year_horizon", "class", "attr37_missing"}
FEATURE_COLS = [c for c in master.columns
                if c not in EXCLUDE_COLS
                and master[c].dtype in [np.float64, np.float32, np.int64, np.int32]]
X = master[FEATURE_COLS].fillna(0.0)
print(f"System model feature matrix: {X.shape[0]:,} rows x {X.shape[1]} features")

# Standardise so VIF reflects correlation structure, not units/scale.
Xz = pd.DataFrame(StandardScaler().fit_transform(X), columns=FEATURE_COLS)

# ---- 2. Variance Inflation Factors ----------------------------------------
# VIF_j = 1 / (1 - R^2_j) from regressing feature j on all other features.
Xc = add_constant(Xz, has_constant="add")
vif_rows = []
for j, col in enumerate(Xc.columns):
    if col == "const":
        continue
    v = variance_inflation_factor(Xc.values, j)
    vif_rows.append({"feature": col, "VIF": v})
vif = (pd.DataFrame(vif_rows)
       .sort_values("VIF", ascending=False)
       .reset_index(drop=True))
# Guard against numerical infinities (perfect collinearity) for reporting.
vif["VIF_display"] = vif["VIF"].replace([np.inf, -np.inf], np.nan)
vif.to_csv(TABLE_DIR + "19_vif.csv", index=False)

n_gt5  = int((vif["VIF"] > 5).sum())
n_gt10 = int((vif["VIF"] > 10).sum())
finite_vif = vif["VIF_display"].dropna()
summary = pd.DataFrame([{
    "n_features"        : len(vif),
    "VIF_gt_5"          : n_gt5,
    "VIF_gt_10"         : n_gt10,
    "VIF_median"        : round(float(finite_vif.median()), 3),
    "VIF_p90"           : round(float(finite_vif.quantile(0.90)), 3),
    "VIF_max_finite"    : round(float(finite_vif.max()), 3),
    "n_VIF_infinite"    : int(vif["VIF_display"].isna().sum()),
}])
summary.to_csv(TABLE_DIR + "19b_collinearity_summary.csv", index=False)
print("\nCollinearity summary:")
print(summary.to_string(index=False))
print(f"\nTop 10 VIF:\n{vif[['feature','VIF']].head(10).to_string(index=False)}")

# ---- 3. Figure 29: top-20 VIF bar (grayscale) -----------------------------
top = vif.head(20).iloc[::-1]
fig, ax = plt.subplots(figsize=(6.4, 6.0))
bars = ax.barh(top["feature"], top["VIF_display"].fillna(top["VIF_display"].max()),
               color=GREY_MID, edgecolor="black", linewidth=0.6)
ax.axvline(5,  color=GREY_DARK, linestyle="--", linewidth=1.0)
ax.axvline(10, color="black",   linestyle="-.", linewidth=1.0)
ax.set_xlabel("Variance Inflation Factor")
ax.set_ylabel("Feature")
ax.set_xscale("log")
savefig(fig, "29_vif_top")

# ---- 4. Figure 30: absolute-correlation clustermap (Greys) ----------------
corr = Xz.corr().abs()
cg = sns.clustermap(corr, cmap="Greys", figsize=(9, 9),
                    vmin=0, vmax=1, xticklabels=False, yticklabels=False,
                    cbar_pos=(0.02, 0.80, 0.03, 0.15))
cg.savefig(FIG_DIR + "30_correlation_clustermap.png", dpi=600, bbox_inches="tight")
cg.savefig(FIG_DIR + "30_correlation_clustermap.pdf", bbox_inches="tight")
plt.close("all")

# ---- 5. High-correlation pair inventory (|r| > 0.8) -----------------------
cu = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
pairs = (cu.stack()
         .reset_index()
         .rename(columns={"level_0": "feature_a", "level_1": "feature_b", 0: "abs_corr"}))
pairs = pairs[pairs["abs_corr"] > 0.8].sort_values("abs_corr", ascending=False)
pairs.to_csv(TABLE_DIR + "19c_high_corr_pairs.csv", index=False)
print(f"\nFeature pairs with |r| > 0.8: {len(pairs)}")
print(pairs.head(10).to_string(index=False))
print("\nNB12 complete: VIF, summary, clustermap, high-corr pairs saved.")


System model feature matrix: 43,405 rows x 64 features



Collinearity summary:
 n_features  VIF_gt_5  VIF_gt_10  VIF_median  VIF_p90  VIF_max_finite  n_VIF_infinite
         64        47         36      14.653  140.748      516664.948               0

Top 10 VIF:
                         feature           VIF
                          Attr14 516664.948365
                  ebit_to_assets 357416.251643
                          Attr18  15588.002890
book_value_equity_to_liabilities    277.030671
                          Attr17    271.283155
                          Attr16    151.969887
                          Attr19    140.795188
                          Attr26    140.636327
                equity_to_assets     97.500283
     total_liabilities_to_assets     94.225295



Feature pairs with |r| > 0.8: 65
                       feature_a        feature_b  abs_corr
                  ebit_to_assets           Attr14  0.999979
                          Attr14           Attr18  0.999516
                  ebit_to_assets           Attr18  0.999300
book_value_equity_to_liabilities           Attr17  0.996898
                          Attr16           Attr26  0.994097
     total_liabilities_to_assets equity_to_assets  0.987502
                          Attr19           Attr23  0.986928
            net_profit_to_assets   ebit_to_assets  0.982429
            net_profit_to_assets           Attr14  0.982411
            net_profit_to_assets           Attr18  0.981730

NB12 complete: VIF, summary, clustermap, high-corr pairs saved.
